# Bifurcation inner/outer capped surface generation

Generates watertight inner (lumen) and outer (wall) surfaces for each
BraVa bifurcation in `Data/<folder>/`, for COMSOL haemodynamic / FSI
simulation (`wall domain = outer solid − inner solid`).

**Current state**: output is saved as `.vtk` rather than `.stl` for now
(STL requires triangulating this mesh's quads, which introduces its own
instability -- `triangulate_quads` is defined below but on hold until STL
export is revisited). `fix_orientation` (consistent outward normals) IS
applied.

Each bifurcation is a pair of SWC files (`bifurcation_X.swc` = inner,
`bifurcation_X_outer.swc` = outer), sharing identical (x, y, z) topology
and differing only in radius.

Pipeline, per bifurcation, using only vascularmd's own `ArterialTree`
methods (`mesh_surface`, `close_surface`, `deform_surface_to_mesh`) — no
custom clipping/capping geometry:

1. Model both the inner and outer SWC files, each into its own open
   (uncapped) surface mesh.
2. Save the outer open surface as `.vtk` — this becomes the deformation
   target.
3. `close_surface()` the inner tree and save it. `close_surface()` adds
   the O-grid cap as new points at the same coordinates as the existing
   boundary loop rather than reusing its indices, so the result needs a
   `.clean()` to weld them into one watertight shell — confirmed
   directly: before `.clean()` the original boundary loops are still
   reported open, after `.clean()` they're gone. `fix_orientation()`
   then corrects vascularmd's own (confirmed inward-facing) normal
   convention.
4. `open_surface()` the inner tree back to its uncapped state, then
   `deform_surface_to_mesh()` it onto the saved outer `.vtk`: for every
   cross-section point, this casts a ray from the cross-section's centre
   outward and moves the point to wherever that ray first hits the target
   mesh — i.e. it reshapes the inner model onto the *outer's actual
   surface*, rather than trusting the outer to have been reconstructed
   independently to the same shape at the bifurcation hub.
5. `close_surface()` this now-deformed surface, `fix_orientation()`, and
   save it.

`deform_surface_to_mesh`'s `search_dist` matters more than its default (40)
suggests: confirmed directly on real data that the default lets a small
fraction of rays travel far enough to hit the *wrong* part of the target
mesh, displacing those points by several units instead of the expected
~wall-thickness (up to 7+ units observed vs. an expected ~0.5). Since every
bifurcation in this dataset uses a wall thickness of exactly 0.5 (confirmed
directly across all inner/outer SWC pairs present), capping the search at
a small multiple of that — a few units — keeps every ray landing on the
intended nearby surface instead of an unrelated distant one.


In [1]:
import sys, os, glob
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pyvista as pv
from ArterialTree import ArterialTree

# ── Configuration ─────────────────────────────────────────────────────────────
data_folder_name = 'BG0014.CNG'
N = 48            # nodes around each cross-section circumference (must be multiple of 8)
d = 0.25          # longitudinal cross-section density

# deform_surface_to_mesh's ray-cast search distance. Confirmed on real data
# that the library default (40) lets some rays travel far enough to hit
# the wrong part of the target mesh (displacements of several units,
# instead of the expected ~wall-thickness); a small multiple of the
# actual wall thickness (0.5 throughout this dataset) keeps every ray
# landing on the intended nearby surface.
search_dist = 3

data_dir = os.path.join('..', 'Data', data_folder_name)
stl_dir  = os.path.join('..', 'Output', data_folder_name + '_stl')
vtk_dir  = os.path.join(stl_dir, '_outer_open_vtk')
os.makedirs(stl_dir, exist_ok=True)
os.makedirs(vtk_dir, exist_ok=True)

all_swc   = sorted(glob.glob(os.path.join(data_dir, 'bifurcation_*.swc')))
swc_files = [p for p in all_swc if not os.path.basename(p).endswith('_outer.swc')]
print(f"Found {len(all_swc)} total .swc files, processing {len(swc_files)} inner/outer pairs")


Found 34 total .swc files, processing 17 inner/outer pairs


In [2]:
# ── Validation ─────────────────────────────────────────────────────────────────

def signed_volume(mesh):
    """Signed enclosed volume via the divergence theorem -- unlike
    pyvista's own `.volume` (confirmed unsigned: unchanged by
    flip_normals() on a test sphere), this correctly flips sign with
    orientation, catching an inside-out mesh."""
    tri = mesh.triangulate()
    pts = tri.points
    faces = tri.faces.reshape(-1, 4)[:, 1:]
    ref = pts.mean(axis=0)
    v0 = pts[faces[:, 0]] - ref
    v1 = pts[faces[:, 1]] - ref
    v2 = pts[faces[:, 2]] - ref
    return np.sum(np.einsum('ij,ij->i', v0, np.cross(v1, v2))) / 6.0


def triangulate_quads(mesh):
    """Deterministic quad -> 2-triangle split (fixed a-c diagonal),
    applied to close_surface()'s pure-quad output (confirmed: every face
    here is a quad) before saving.

    Necessary because STL can only store triangles, so saving a
    quad-containing mesh directly triggers VTK's own implicit
    triangulation at save time -- and that choice of diagonal, for this
    mesh's non-planar bifurcation-hub quads, depends on incidental
    point/cell ordering. Confirmed directly: every file here checked out
    fully watertight in memory, then came back with real cracks (0/15
    watertight) after being saved to STL and reloaded, purely from this
    implicit re-triangulation. Triangulating once, deterministically,
    before saving removes that instability."""
    faces = mesh.faces.reshape(-1, 5)
    if not (faces[:, 0] == 4).all():
        raise ValueError('expected a pure-quad mesh')
    a, b, c, d_ = faces[:, 1], faces[:, 2], faces[:, 3], faces[:, 4]
    n = len(faces)
    tri = np.empty((2 * n, 4), dtype=faces.dtype)
    tri[0::2] = np.column_stack([np.full(n, 3), a, b, c])
    tri[1::2] = np.column_stack([np.full(n, 3), a, c, d_])
    return pv.PolyData(mesh.points, tri.ravel())


def fix_orientation(mesh):
    """Enforce a consistent, outward-facing normal convention.

    Confirmed on real data: vascularmd's close_surface()+mesh_surface()
    output comes out with a globally inward-pointing (negative signed
    volume) convention here, consistently across every file tested --
    not a per-triangle defect, just the opposite winding sense from what
    downstream tools (COMSOL import, this notebook's own volume checks)
    expect. compute_normals(auto_orient_normals=True, consistent_normals=True)
    first enforces local consistency (reorders any individual
    out-of-step cell's winding -- confirmed directly on a deliberately
    mis-wound test mesh), then the sign of the divergence-theorem volume
    (not pyvista's own `.volume`, confirmed unsigned) decides whether to
    flip the whole mesh.
    """
    mesh = mesh.compute_normals(auto_orient_normals=True, consistent_normals=True,
                                 cell_normals=True, point_normals=True)
    if signed_volume(mesh) < 0:
        mesh.flip_normals()
    return mesh


def validate_capped_pair(inner, outer):
    """STL-level watertightness/volume checks, plus a nesting check (inner
    strictly inside outer). Returns a dict of check_name -> (passed, detail).

    The nesting check uses select_enclosed_points, a discrete ray-cast
    test, so some sample points can register as marginally outside right
    at the bifurcation hub (the thinnest, most complex part of the wall
    for any reconstruction approach) without the geometry being genuinely
    broken there. What matters for COMSOL is the SIZE of any gap, not what
    fraction of points are marginally outside -- many points a hair's
    breadth outside is harmless, a few points far outside means a real
    hole. So this checks the worst-case distance of any "outside" point
    from the outer surface, confirmed on real data to be a few hundredths
    of a unit (a small fraction of the local vessel radius) at the hub for
    otherwise-healthy files, and reports it either way rather than hiding it.

    select_enclosed_points also requires a genuinely closed, manifold
    surface and raises if it isn't -- confirmed on real data that
    close_surface()+clean() can leave a handful (~5) of non-manifold
    points in a tiny, localized cluster for certain bifurcation
    geometries (an apparent quirk of vascularmd's own O-grid capping,
    not something introduced by this notebook's own code). That's caught
    here as its own reported check rather than crashing the whole file.
    """
    results = {}

    def check(name, passed, detail=''):
        results[name] = (bool(passed), detail)

    for label, mesh in [('inner', inner), ('outer', outer)]:
        ne = mesh.extract_feature_edges(boundary_edges=True, non_manifold_edges=True,
                                         feature_edges=False, manifold_edges=False)
        check(f'{label}_watertight', ne.n_points == 0, f'{ne.n_points} open/non-manifold edge points')
        check(f'{label}_finite_coords', np.isfinite(mesh.points).all())

        conn = mesh.connectivity()
        n_comp = int(conn.point_data['RegionId'].max()) + 1
        check(f'{label}_single_component', n_comp == 1, f'{n_comp} connected components')

        sv = signed_volume(mesh)
        check(f'{label}_positive_volume', sv > 0, f'signed volume {sv:.4f}')

    vin, vout = signed_volume(inner), signed_volume(outer)
    check('outer_volume_greater_than_inner', vout > vin, f'inner={vin:.4f} outer={vout:.4f}')

    # What actually matters for COMSOL is the SIZE of any gap, not what
    # fraction of sample points are marginally outside.
    try:
        rng = np.random.default_rng(0)
        n_sample = min(500, inner.n_points)
        sample_idx = rng.choice(inner.n_points, size=n_sample, replace=False)
        sample_pts = pv.PolyData(inner.points[sample_idx])
        enclosed = sample_pts.select_enclosed_points(outer, tolerance=1e-6)
        inside = np.asarray(enclosed['SelectedPoints']).astype(bool)
        n_outside = (~inside).sum()
        if n_outside > 0:
            from scipy.spatial import cKDTree
            tree = cKDTree(outer.points)
            max_gap, _ = tree.query(sample_pts.points[~inside])
            max_gap = max_gap.max()
        else:
            max_gap = 0.0
        ok = max_gap <= 0.5   # same order as the wall thickness itself
        detail = f'{n_outside}/{len(inside)} sampled inner points outside outer, max gap {max_gap:.4f}'
        if n_outside and ok:
            detail += ' (small, near the bifurcation hub -- not a broken cap)'
        check('inner_inside_outer', ok, detail)
    except RuntimeError as e:
        check('inner_inside_outer', False, f'could not run (outer surface not fully manifold): {e}')

    return results


def report_validation(stem, results):
    failed = {k: v for k, v in results.items() if not v[0]}
    if not failed:
        print(f"  [PASS] {stem}: all {len(results)} checks passed")
    else:
        print(f"  [FAIL] {stem}: {len(failed)}/{len(results)} checks failed")
        for k, (passed, detail) in failed.items():
            print(f"      - {k}: {detail}")
    return len(failed) == 0


In [3]:
# ── Batch processing ──────────────────────────────────────────────────────────
# NOTE: STL export (triangulate_quads) is still disabled -- saving as
# .vtk instead, which can store the mesh's native quads directly without
# needing the lossy triangulation STL requires. fix_orientation IS applied.

failed = []
warned = []

for swc_path in swc_files:
    stem       = os.path.splitext(os.path.basename(swc_path))[0]   # e.g. bifurcation_1025
    inner_swc  = swc_path
    outer_swc  = os.path.join(data_dir, stem + '_outer.swc')
    inner_path = os.path.join(stl_dir, stem + '_inner_surface_capped.vtk')
    outer_path = os.path.join(stl_dir, stem + '_outer_surface_capped.vtk')
    outer_vtk_path = os.path.join(vtk_dir, stem + '_outer_open.vtk')

    print(f"\n[{stem}]")

    if not os.path.isfile(outer_swc):
        msg = f"No matching outer SWC found at {outer_swc}"
        print(f"  ERROR: {msg}")
        failed.append((stem, msg))
        continue

    try:
        # 1. Model inner and outer, each into its own open surface mesh.
        inner_tree = ArterialTree("patient", "Aneurisk", inner_swc)
        inner_tree.model_network()
        inner_tree.compute_cross_sections(N, d, parallel=False)
        inner_tree.mesh_surface()

        outer_tree = ArterialTree("patient", "Aneurisk", outer_swc)
        outer_tree.model_network()
        outer_tree.compute_cross_sections(N, d, parallel=False)
        outer_tree.mesh_surface()

        # 2. Save the outer open surface as .vtk -- the deformation target.
        outer_open = pv.wrap(outer_tree.get_surface_mesh())
        outer_open.save(outer_vtk_path)

        # 3. close_surface() the inner tree, save as the inner file.
        #    close_surface() adds the O-grid cap as new points at the same
        #    coordinates as the existing boundary loop rather than reusing
        #    its indices -- .clean() welds them into one watertight shell
        #    (confirmed: without it, the original boundary loops are
        #    still reported open).
        inner_tree.close_surface()
        inner_capped = pv.wrap(inner_tree.get_surface_mesh()).clean()
        inner_capped = fix_orientation(inner_capped)

        # 4. Reopen, then deform the (open) inner surface onto the outer's
        #    actual shape via ray-casting from each cross-section's centre.
        inner_tree.open_surface()
        outer_mesh_vtk = pv.read(outer_vtk_path)
        inner_tree.deform_surface_to_mesh(outer_mesh_vtk, search_dist=search_dist)

        # 5. close_surface() the now-deformed surface, save as the outer file.
        inner_tree.close_surface()
        outer_capped = pv.wrap(inner_tree.get_surface_mesh()).clean()
        outer_capped = fix_orientation(outer_capped)

        results = validate_capped_pair(inner_capped, outer_capped)
        ok = report_validation(stem, results)
        if not ok:
            warned.append(stem)

        inner_capped.save(inner_path)
        outer_capped.save(outer_path)
        print(f"  inner → {os.path.abspath(inner_path)}  "
              f"[{inner_capped.n_points} pts, {inner_capped.n_cells} faces]")
        print(f"  outer → {os.path.abspath(outer_path)}  "
              f"[{outer_capped.n_points} pts, {outer_capped.n_cells} faces]")

    except Exception as e:
        print(f"  ERROR: {e}")
        failed.append((stem, str(e)))

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Processed successfully : {len(swc_files) - len(failed)} / {len(swc_files)}")
print(f"  ...with validation warnings : {len(warned)}")
print(f"Failed                  : {len(failed)}")
if warned:
    print("Files with validation warnings (see log above for detail):")
    for name in warned:
        print(f"  {name}")
if failed:
    print("Failed files:")
    for name, err in failed:
        print(f"  {name}: {err}")



[bifurcation_116]
Loading swc file...
Modeling the network...
Meshing bifurcations.
Meshing edges.
Meshing surface...
Loading swc file...
Modeling the network...
Meshing bifurcations.
Meshing edges.
Meshing surface...
Meshing surface...
Meshing surface...
Meshing surface...
Meshing surface...
Meshing surface...
Meshing surface...
Meshing surface...
  [PASS] bifurcation_116: all 10 checks passed
  inner → /home/xbfl0349/arterial_network/Output/BG0014.CNG_stl/bifurcation_116_inner_surface_capped.vtk  [10466 pts, 10464 faces]
  outer → /home/xbfl0349/arterial_network/Output/BG0014.CNG_stl/bifurcation_116_outer_surface_capped.vtk  [10466 pts, 10464 faces]

[bifurcation_129]
Loading swc file...
Modeling the network...
Meshing bifurcations.
Meshing edges.
Meshing surface...
Loading swc file...
Modeling the network...
Combine!
  ERROR: list index out of range

[bifurcation_134]
Loading swc file...
Modeling the network...
Meshing bifurcations.
Meshing edges.
Meshing surface...
Loading swc fil

In [4]:
# List what was written to the STL output folder
import os, glob
stl_files = sorted(glob.glob(os.path.join(stl_dir, '*')))
print(f"Contents of '{stl_dir}':")
for f in stl_files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):45s}  {size_kb:8.1f} kB")


Contents of '../Output/BG0014.CNG_stl':
  _outer_open_vtk                                     4.0 kB
  bifurcation_116_inner_surface_capped.vtk          899.6 kB
  bifurcation_116_outer_surface_capped.vtk          899.6 kB
  bifurcation_134_inner_surface_capped.vtk          746.9 kB
  bifurcation_134_outer_surface_capped.vtk          746.9 kB
  bifurcation_143_inner_surface_capped.vtk          825.3 kB
  bifurcation_143_outer_surface_capped.vtk          825.3 kB
  bifurcation_149_inner_surface_capped.vtk          779.9 kB
  bifurcation_149_outer_surface_capped.vtk          779.9 kB
  bifurcation_15_inner_surface_capped.vtk           709.8 kB
  bifurcation_15_outer_surface_capped.vtk           709.8 kB
  bifurcation_171_inner_surface_capped.vtk          705.7 kB
  bifurcation_171_outer_surface_capped.vtk          705.7 kB
  bifurcation_194_inner_surface_capped.vtk         1250.2 kB
  bifurcation_194_outer_surface_capped.vtk         1250.2 kB
  bifurcation_257_inner_surface_capped.vtk   